In [ ]:
# Adversarial Defense Self Training 

import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.validation import check_random_state, check_X_y, check_array, check_is_fitted
from scipy.stats import loguniform
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt

class FiniteAttackModel:
    def __init__(self, f_attack=0.5, epsilon=0.1, random_state=None):
        self.f_attack = f_attack
        self.epsilon = epsilon
        self.random_state = random_state

    def attack(self, X, y_pseudo, model):
        if not hasattr(model, 'coef_'):
            return X.copy()

        X_adv = X.copy()
        w = model.coef_[0]
        b = model.intercept_
        rng = check_random_state(self.random_state)

        for i in range(len(X)):
            x = X[i]
            y_i = y_pseudo[i]
            margin = y_i * (np.dot(w, x) + b)

            if margin > 0:  # Only attack correctly classified samples
                x_a = x - (2 * margin / np.dot(w, w)) * w * y_i  # Target point
                norm_sum = np.linalg.norm(x_a) + np.linalg.norm(x)
                rel_dist = np.linalg.norm(x_a - x) / norm_sum if norm_sum > 0 else 0
                max_delta = (1 - rel_dist * self.f_attack) * (x_a - x)
                delta = np.clip(max_delta, -self.epsilon, self.epsilon)
                X_adv[i] = x + delta

        return X_adv

class RobustAD_ST(BaseEstimator, ClassifierMixin):
    def __init__(self, C1=1.0, C2=0.1, lambda_adv=0.5, rho=0.1, mu=0.1,
                 f_attack=0.6, n_iter=10, kernel='linear', epsilon=0.1,
                 confidence_threshold=0.7, min_confidence_diff=0.2,
                 early_stopping_rounds=3, random_state=None):
        self.C1 = C1
        self.C2 = C2
        self.lambda_adv = lambda_adv
        self.rho = rho
        self.mu = mu
        self.f_attack = f_attack
        self.n_iter = n_iter
        self.kernel = kernel
        self.epsilon = epsilon
        self.confidence_threshold = confidence_threshold
        self.min_confidence_diff = min_confidence_diff
        self.early_stopping_rounds = early_stopping_rounds
        self.random_state = random_state

        self.model = None
        self.attack_model = None
        self.intercept_ = None
        self.coef_ = None
        self.support_vectors_ = None
        self.n_support_ = None
        self.best_loss = np.inf
        self.no_improvement_count = 0

    def _tsvm_loss(self, X, y, labeled_mask):
        if not hasattr(self.model, 'coef_'):
            return 0.0

        w = self.model.coef_[0]
        b = self.intercept_
        margins = y * (X.dot(w) + b)
        hinge_loss = np.maximum(0, 1 - margins)
        labeled_loss = np.sum(hinge_loss[labeled_mask])
        unlabeled_loss = np.sum(hinge_loss[~labeled_mask])
        reg_loss = 0.5 * np.dot(w, w)
        return reg_loss + self.C1 * labeled_loss + self.C2 * unlabeled_loss

    def _adv_loss(self, X_adv, X_orig):
        delta = X_adv - X_orig
        feature_mins = np.min(X_orig, axis=0)
        feature_maxs = np.max(X_orig, axis=0)
        lower_bounds = self.f_attack * (feature_mins - X_orig)
        upper_bounds = self.f_attack * (feature_maxs - X_orig)
        clipped_delta = np.maximum(np.minimum(delta, upper_bounds), lower_bounds)
        return np.sum((delta - clipped_delta) ** 2)

    def _pseudo_loss(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)

    def _finite_loss(self, X_adv, X_orig):
        delta = X_adv - X_orig
        norm_orig = np.linalg.norm(X_orig, axis=1, keepdims=True)
        norm_adv = np.linalg.norm(X_adv, axis=1, keepdims=True)
        safe_norms = np.where(norm_orig + norm_adv == 0, 1e-10, norm_orig + norm_adv)
        rel_dists = np.linalg.norm(delta, axis=1, keepdims=True) / safe_norms
        return np.sum((1 - rel_dists * self.f_attack) * np.linalg.norm(delta, axis=1))

    def _feature_robustness_loss(self, X_adv, X_orig):
        """Additional loss term for feature-level robustness"""
        feature_variances = np.var(X_orig, axis=0)
        weighted_diffs = np.sum((X_adv - X_orig)**2 / (feature_variances + 1e-10), axis=1)
        return np.mean(weighted_diffs)

    def _get_high_confidence_samples(self, X_unlabeled, decision_values):
        """Improved high-confidence sample selection"""
        pseudo_labels = np.sign(decision_values)
        confidences = np.abs(decision_values)

        # Only select samples where confidence is above threshold and significantly different from 0
        high_conf_mask = (confidences > self.confidence_threshold) & \
                         (confidences > self.min_confidence_diff)

        return X_unlabeled[high_conf_mask], pseudo_labels[high_conf_mask]

    def fit(self, X_labeled, y_labeled, X_unlabeled):
        rng = check_random_state(self.random_state)
        self.attack_model = FiniteAttackModel(
            f_attack=self.f_attack,
            epsilon=self.epsilon,
            random_state=self.random_state
        )

        # Initialize with labeled data
        self.model = SVC(
            kernel=self.kernel,
            C=self.C1,
            random_state=self.random_state
        )
        self.model.fit(X_labeled, y_labeled)
        self._update_model_attributes()

        n_labeled = len(X_labeled)
        X_combined = np.vstack([X_labeled, X_unlabeled])
        y_combined = np.hstack([y_labeled, np.zeros(len(X_unlabeled))])
        labeled_mask = np.zeros(len(X_combined), dtype=bool)
        labeled_mask[:n_labeled] = True

        for iteration in range(self.n_iter):
            # Generate pseudo-labels with improved confidence estimation
            decision_values = self.model.decision_function(X_unlabeled)
            X_high_conf, pseudo_labels_high_conf = self._get_high_confidence_samples(
                X_unlabeled, decision_values)

            if len(X_high_conf) == 0:
                continue  # Skip iteration if no high-confidence samples

            # Generate adversarial samples
            X_adv = self.attack_model.attack(X_high_conf, pseudo_labels_high_conf, self.model)

            # Update combined data
            X_combined = np.vstack([X_labeled, X_adv])
            y_combined = np.hstack([y_labeled, pseudo_labels_high_conf])
            labeled_mask = np.zeros(len(X_combined), dtype=bool)
            labeled_mask[:n_labeled] = True

            # Compute total loss with additional robustness terms
            total_loss = (
                self._tsvm_loss(X_combined, y_combined, labeled_mask) +
                self.lambda_adv * self._adv_loss(X_adv, X_high_conf) +
                self.rho * self._pseudo_loss(pseudo_labels_high_conf,
                                           self.model.predict(X_high_conf)) +
                self.mu * self._finite_loss(X_adv, X_high_conf) +
                0.1 * self._feature_robustness_loss(X_adv, X_high_conf)  # Additional robustness term
            )

            # Early stopping check
            if total_loss < self.best_loss - 1e-5:
                self.best_loss = total_loss
                self.no_improvement_count = 0
            else:
                self.no_improvement_count += 1
                if self.no_improvement_count >= self.early_stopping_rounds:
                    break

            # Retrain model
            self.model.fit(X_combined, y_combined)
            self._update_model_attributes()

        return self

    def _update_model_attributes(self):
        """Helper method to update model attributes"""
        self.intercept_ = self.model.intercept_[0]
        self.coef_ = self.model.coef_
        self.support_vectors_ = self.model.support_vectors_
        self.n_support_ = self.model.n_support_

    def predict(self, X):
        check_is_fitted(self)
        X = check_array(X)
        return self.model.predict(X)

    def decision_function(self, X):
        check_is_fitted(self)
        X = check_array(X)
        return self.model.decision_function(X)

def load_and_prepare_data(dataset_id=94, test_size=0.3, unlabeled_ratio=0.7, random_state=42):
    dataset = fetch_ucirepo(id=dataset_id)
    X, y = dataset.data.features, dataset.data.targets.values.ravel()
    y = np.where(y == y[0], 1, -1)  # Ensure binary labels

    # Split into labeled and unlabeled
    X_labeled, X_unlabeled, y_labeled, _ = train_test_split(
        X, y, test_size=unlabeled_ratio, random_state=random_state)
    X_train, X_test, y_train, y_test = train_test_split(
        X_labeled, y_labeled, test_size=test_size, random_state=random_state)

    # Standardize
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_unlabeled = scaler.transform(X_unlabeled)
    X_test = scaler.transform(X_test)

    return X_train, X_test, X_unlabeled, y_train, y_test, scaler

def evaluate_model(model, X_test, y_test, attack_model=None):
    if attack_model is not None:
        X_test_attacked = attack_model.attack(X_test, model.predict(X_test), model)
        y_pred = model.predict(X_test_attacked)
    else:
        y_pred = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }

def plot_results(results):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
    models = list(results.keys())

    fig, ax = plt.subplots(figsize=(12, 6))
    width = 0.2
    x = np.arange(len(models))

    for i, metric in enumerate(metrics):
        ax.bar(x + i*width, [results[model][metric] for model in models],
               width, label=metric)

    ax.set_ylabel('Score')
    ax.set_title('Model Performance Comparison')
    ax.set_xticks(x + 1.5*width)
    ax.set_xticklabels(models)
    ax.legend()
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.show()

def main():
    # Load data
    X_train, X_test, X_unlabeled, y_train, y_test, _ = load_and_prepare_data()

    # Baseline SVM (supervised)
    svm = SVC(kernel='linear', C=1.0, random_state=42).fit(X_train, y_train)

    # Robust AD-ST with hyperparameter tuning (same parameters as before)
    param_dist = {
        'C1': loguniform(1e-2, 1e1),
        'C2': loguniform(1e-3, 1e0),
        'lambda_adv': [0.2, 0.4, 0.6, 0.8],
        'rho': [0.05, 0.1, 0.2],
        'mu': [0.05, 0.1, 0.2],
        'f_attack': [0.3, 0.5, 0.7],
        'n_iter': [5, 10, 15],
        'confidence_threshold': [0.6, 0.7, 0.8]
    }

    robust_ad_st = RobustAD_ST(random_state=42)
    random_search = RandomizedSearchCV(
        robust_ad_st, param_dist, n_iter=20, cv=3, scoring='accuracy',
        n_jobs=-1, verbose=2, random_state=42
    )
    random_search.fit(X_train, y_train, X_unlabeled=X_unlabeled)
    best_robust_ad_st = random_search.best_estimator_

    # Attack model
    attack_model = FiniteAttackModel(f_attack=0.5, epsilon=0.1)

    # Evaluate
    results = {
        "SVM (Clean)": evaluate_model(svm, X_test, y_test),
        "SVM (Attacked)": evaluate_model(svm, X_test, y_test, attack_model),
        "Robust AD-ST (Attacked)": evaluate_model(best_robust_ad_st, X_test, y_test, attack_model)
    }

    print("\nPerformance Results:")
    for name, metrics in results.items():
        print(f"\n{name}:")
        print(f"  Accuracy: {metrics['Accuracy']:.4f}")
        print(f"  Precision: {metrics['Precision']:.4f}")
        print(f"  Recall: {metrics['Recall']:.4f}")
        print(f"  F1-Score: {metrics['F1']:.4f}")

    # Plot results
    plot_results(results)

    return X_test, y_test, svm, best_robust_ad_st, attack_model

if __name__ == "__main__":
    X_test, y_test, svm, best_robust_ad_st, attack_model = main()

In [ ]:
# TSNE Visualisations

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

def plot_robust_tsne_visualization(X_test, y_test, svm, robust_model, attack_model):
    # Create attacked version of the test set
    X_test_attacked = attack_model.attack(X_test, svm.predict(X_test), svm)

    # Combine all data for consistent TSNE transformation
    all_data = np.vstack([X_test, X_test_attacked])

    # Apply TSNE (you can adjust perplexity based on your dataset size)
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    tsne_results = tsne.fit_transform(all_data)

    # Split back into original and attacked
    X_tsne = tsne_results[:len(X_test)]
    X_tsne_attacked = tsne_results[len(X_test):]

    # Create figure
    plt.figure(figsize=(18, 6))

    # Colormaps
    cm_bright = ListedColormap(['#FF0000', '#0000FF'])  # Red for -1, Blue for 1

    # Get predictions for visualization
    svm_clean_pred = svm.predict(X_test)
    svm_attacked_pred = svm.predict(X_test_attacked)
    robust_attacked_pred = robust_model.predict(X_test_attacked)

    # Plot SVM clean predictions
    plt.subplot(1, 3, 1)
    plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=svm_clean_pred, cmap=cm_bright, alpha=0.6, edgecolors='k')
    plt.title(f"Standard SVM on Clean Data")
    plt.xlabel("TSNE 1")
    plt.ylabel("TSNE 2")

    # Plot SVM attacked predictions
    plt.subplot(1, 3, 2)
    plt.scatter(X_tsne_attacked[:, 0], X_tsne_attacked[:, 1], c=svm_attacked_pred, cmap=cm_bright, alpha=0.6, edgecolors='k')
    plt.title(f"Standard SVM on Attacked Data")
    plt.xlabel("TSNE 1")
    plt.ylabel("TSNE 2")

    # Plot RobustAD-ST attacked predictions
    plt.subplot(1, 3, 3)
    plt.scatter(X_tsne_attacked[:, 0], X_tsne_attacked[:, 1], c=robust_attacked_pred, cmap=cm_bright, alpha=0.6, edgecolors='k')
    plt.title(f"RobustAD-ST on Attacked Data")
    plt.xlabel("TSNE 1")
    plt.ylabel("TSNE 2")

    plt.tight_layout()
    plt.show()

# Run the visualization after your main code
plot_robust_tsne_visualization(X_test, y_test, svm, best_robust_ad_st, attack_model)